# Thrasher Attention Profile — p=59, seed=485

Pre-intervention characterization for REQ_068.

Questions:
1. Which failure mode? Competition (too many frequencies) vs. weak signal (winners present but faint)
2. How long does the competition window run?
3. Per-head breakdown at plateau onset — evenly dispersed or concentrated?
4. Will reduction in competition allow Freq 21 to build MLP mass?

Comparison variant: p=59, seed=999 (healthy — 3 clear frequency winners by epoch 1500)

In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
import json
from pathlib import Path

RESULTS = Path('../results/modulo_addition_1layer')
THRASHER_DIR  = RESULTS / 'modulo_addition_1layer_p59_seed485_dseed598'
HEALTHY_DIR   = RESULTS / 'modulo_addition_1layer_p59_seed999_dseed598'

PLATEAU_EPOCH = 1500   # train loss ~0 for Thrasher
FREQ_LABELS   = list(range(1, 30))  # p=59 → frequencies 1..29

In [2]:
def load_attention_trajectory(variant_dir: Path) -> tuple[list[int], np.ndarray]:
    """Load QK^T Fourier fractions for all epochs.
    
    Returns:
        epochs: sorted list of epoch numbers
        qk_mean: (n_epochs, n_freqs) — mean across 4 heads
    """
    attn_dir = variant_dir / 'artifacts' / 'attention_fourier'
    files = sorted(attn_dir.glob('epoch_*.npz'), key=lambda p: int(p.stem.split('_')[1]))
    epochs = [int(p.stem.split('_')[1]) for p in files]
    qk_mean = np.stack(
        [np.load(f)['qk_freq_norms'].mean(axis=0) for f in files]
    )  # (n_epochs, n_freqs)
    return epochs, qk_mean


def load_attention_per_head(variant_dir: Path) -> tuple[list[int], np.ndarray]:
    """Load QK^T Fourier fractions per head for all epochs.
    
    Returns:
        epochs: sorted list of epoch numbers
        qk_heads: (n_epochs, n_heads, n_freqs)
    """
    attn_dir = variant_dir / 'artifacts' / 'attention_fourier'
    files = sorted(attn_dir.glob('epoch_*.npz'), key=lambda p: int(p.stem.split('_')[1]))
    epochs = [int(p.stem.split('_')[1]) for p in files]
    qk_heads = np.stack(
        [np.load(f)['qk_freq_norms'] for f in files]
    )  # (n_epochs, n_heads, n_freqs)
    return epochs, qk_heads


def load_neuron_counts(variant_dir: Path, threshold: float = 0.5) -> tuple[list[int], dict[int, list[int]]]:
    """Load committed neuron counts per frequency from neuron_dynamics artifacts.
    
    Returns:
        epochs: sorted list of epoch numbers  
        counts: dict mapping freq_idx (1-based) -> list of committed counts per epoch
    """
    nd_dir = variant_dir / 'artifacts' / 'neuron_dynamics'
    files = sorted(nd_dir.glob('epoch_*.npz'), key=lambda p: int(p.stem.split('_')[1]))
    if not files:
        return [], {}
    epochs = [int(p.stem.split('_')[1]) for p in files]
    
    # Load peak frequencies per neuron per epoch — shape (n_epochs, d_mlp)
    peak_freqs = np.stack([np.load(f)['peak_freq'] for f in files])  # (n_epochs, d_mlp)
    # Load dominant fraction per neuron per epoch — shape (n_epochs, d_mlp)
    dom_frac   = np.stack([np.load(f)['dominant_fraction'] for f in files])  # (n_epochs, d_mlp)
    
    # Count committed neurons per frequency at each epoch
    unique_freqs = np.unique(peak_freqs)
    counts = {}
    for f in unique_freqs:
        committed = ((peak_freqs == f) & (dom_frac >= threshold)).sum(axis=1)  # (n_epochs,)
        counts[int(f)] = committed.tolist()
    return epochs, counts


def load_train_losses(variant_dir: Path) -> tuple[list[int], list[float]]:
    with open(variant_dir / 'metadata.json') as f:
        meta = json.load(f)
    epochs = list(range(len(meta['train_losses'])))
    return epochs, meta['train_losses']

In [3]:
t_epochs, t_qk_mean = load_attention_trajectory(THRASHER_DIR)
h_epochs, h_qk_mean = load_attention_trajectory(HEALTHY_DIR)

print(f"Thrasher: {len(t_epochs)} epochs, {t_qk_mean.shape[1]} frequencies")
print(f"Healthy:  {len(h_epochs)} epochs, {h_qk_mean.shape[1]} frequencies")

Thrasher: 116 epochs, 29 frequencies
Healthy:  95 epochs, 29 frequencies


## 1. Attention Frequency Trajectories — Thrasher vs Healthy

Full trajectory of mean QK^T fraction per frequency for both variants.
Vertical line marks plateau onset (epoch 1500).

In [4]:
# Identify the frequencies worth plotting — those that ever exceed a threshold
PLOT_THRESHOLD = 0.07   # matches multistream attention floor

t_active = np.where(t_qk_mean.max(axis=0) >= PLOT_THRESHOLD)[0]  # freq indices
h_active = np.where(h_qk_mean.max(axis=0) >= PLOT_THRESHOLD)[0]

# Winners and losers in Thrasher
THRASHER_WINNERS    = [5, 21]       # freq labels (1-based) that survive to end
THRASHER_LOST       = [2]           # won but later lost
THRASHER_LOSERS     = [4, 10, 12, 17, 22]  # never win

def freq_color(freq_label: int) -> str:
    if freq_label in THRASHER_WINNERS:
        return 'rgba(80, 200, 120, 0.9)'   # green — stable winner
    if freq_label in THRASHER_LOST:
        return 'rgba(255, 200, 50, 0.9)'   # yellow — transient winner
    return 'rgba(180, 80, 80, 0.6)'        # red — loser


fig = sp.make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=['Thrasher (p=59, seed=485)', 'Healthy (p=59, seed=999)'],
    vertical_spacing=0.10,
)

for fi in t_active:
    freq_label = FREQ_LABELS[fi]
    is_winner = freq_label in THRASHER_WINNERS + THRASHER_LOST
    fig.add_trace(go.Scatter(
        x=t_epochs, y=t_qk_mean[:, fi],
        name=f'freq {freq_label}',
        line=dict(color=freq_color(freq_label), width=2.5 if is_winner else 1.0,
                  dash='solid' if is_winner else 'dot'),
        legendgroup=f'freq{freq_label}',
        showlegend=True,
    ), row=1, col=1)

for fi in h_active:
    freq_label = FREQ_LABELS[fi]
    fig.add_trace(go.Scatter(
        x=h_epochs, y=h_qk_mean[:, fi],
        name=f'freq {freq_label}',
        line=dict(color='rgba(100, 160, 255, 0.85)', width=2),
        legendgroup=f'freq{freq_label}_h',
        showlegend=True,
    ), row=2, col=1)

# Plateau onset marker
for row in (1, 2):
    fig.add_vline(x=PLATEAU_EPOCH, line=dict(color='white', width=1.5, dash='dash'),
                  row=row, col=1)  # type: ignore[arg-type]

fig.update_yaxes(title_text='Mean QK^T fraction', row=1, col=1)
fig.update_yaxes(title_text='Mean QK^T fraction', row=2, col=1)
fig.update_xaxes(title_text='Epoch', row=2, col=1)
fig.update_layout(
    title='Attention Frequency Trajectories — Thrasher vs Healthy',
    height=700,
    template='plotly_dark',
    plot_bgcolor='#0a1010',
    paper_bgcolor='#060a0a',
    font=dict(family='IBM Plex Mono, monospace', size=11),
)
fig.show()

## 2. Per-Head Breakdown at Plateau Onset

Are all 4 heads competing equally, or is one head already leaning toward the winning frequencies?
Heatmap: head × frequency, QK^T fraction at epoch 1500.

In [5]:
_, t_qk_heads = load_attention_per_head(THRASHER_DIR)
_, h_qk_heads = load_attention_per_head(HEALTHY_DIR)

def epoch_idx(epochs: list[int], target: int) -> int:
    return min(range(len(epochs)), key=lambda i: abs(epochs[i] - target))

t_plateau_idx = epoch_idx(t_epochs, PLATEAU_EPOCH)
h_plateau_idx = epoch_idx(h_epochs, PLATEAU_EPOCH)

t_plateau_heads = t_qk_heads[t_plateau_idx]   # (4, 29)
h_plateau_heads = h_qk_heads[h_plateau_idx]   # (4, 29)

fig = sp.make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Thrasher — epoch {t_epochs[t_plateau_idx]}',
        f'Healthy  — epoch {h_epochs[h_plateau_idx]}',
    ],
    horizontal_spacing=0.12,
)

for col, (data, label) in enumerate([(t_plateau_heads, 'Thrasher'), (h_plateau_heads, 'Healthy')], start=1):
    fig.add_trace(go.Heatmap(
        z=data,
        x=[f'k={f}' for f in FREQ_LABELS],
        y=[f'H{i}' for i in range(4)],
        colorscale='Viridis',
        zmin=0, zmax=data.max(),
        colorbar=dict(title='QK^T frac', x=0.46 if col == 1 else 1.0),
        name=label,
    ), row=1, col=col)

fig.update_layout(
    title=f'Per-Head QK^T Fractions at Plateau Onset (epoch {PLATEAU_EPOCH})',
    height=320,
    template='plotly_dark',
    plot_bgcolor='#0a1010',
    paper_bgcolor='#060a0a',
    font=dict(family='IBM Plex Mono, monospace', size=11),
)
fig.show()

print('\nThrasher per-head fractions at plateau onset:')
for h in range(4):
    top = sorted(enumerate(t_plateau_heads[h]), key=lambda x: -x[1])[:5]
    print(f'  H{h}: ' + ', '.join(f'k={FREQ_LABELS[fi]}:{v:.3f}' for fi, v in top))

print('\nHealthy per-head fractions at plateau onset:')
for h in range(4):
    top = sorted(enumerate(h_plateau_heads[h]), key=lambda x: -x[1])[:5]
    print(f'  H{h}: ' + ', '.join(f'k={FREQ_LABELS[fi]}:{v:.3f}' for fi, v in top))


Thrasher per-head fractions at plateau onset:
  H0: k=22:0.125, k=10:0.100, k=21:0.086, k=5:0.086, k=23:0.076
  H1: k=12:0.123, k=5:0.097, k=19:0.071, k=23:0.070, k=4:0.061
  H2: k=5:0.129, k=19:0.087, k=22:0.081, k=28:0.069, k=23:0.059
  H3: k=22:0.089, k=12:0.076, k=23:0.071, k=5:0.065, k=3:0.057

Healthy per-head fractions at plateau onset:
  H0: k=15:0.092, k=5:0.089, k=26:0.074, k=17:0.061, k=19:0.061
  H1: k=15:0.166, k=10:0.070, k=9:0.069, k=28:0.068, k=5:0.062
  H2: k=15:0.124, k=5:0.083, k=21:0.082, k=2:0.074, k=11:0.066
  H3: k=5:0.132, k=12:0.086, k=9:0.064, k=18:0.058, k=28:0.058


## 3. Competitor Energy Decay

Sum of QK^T fractions across non-winning frequencies over time.
This shows how long the competition window runs — and whether it ever fully resolves.

Competitor set (Thrasher): freq 4, 10, 12, 17, 22

In [6]:
COMPETITOR_FREQS = [4, 10, 12, 17, 22]   # never win in Thrasher
WINNER_FREQS     = [5, 21]               # stable winners
LOST_FREQS       = [2]                   # transient winner

def freq_energy(qk_mean: np.ndarray, freq_labels: list[int]) -> np.ndarray:
    """Sum QK^T fractions across specified frequency labels."""
    indices = [i for i, f in enumerate(FREQ_LABELS) if f in freq_labels]
    return qk_mean[:, indices].sum(axis=1)

t_competitor_energy = freq_energy(t_qk_mean, COMPETITOR_FREQS)
t_winner_energy     = freq_energy(t_qk_mean, WINNER_FREQS)
t_lost_energy       = freq_energy(t_qk_mean, LOST_FREQS)
h_winner_energy     = freq_energy(h_qk_mean, [5, 15, 21])   # healthy winners

fig = go.Figure()

fig.add_trace(go.Scatter(x=t_epochs, y=t_competitor_energy,
    name='Thrasher — competitors (4,10,12,17,22)',
    line=dict(color='rgba(220, 80, 80, 0.9)', width=2)))

fig.add_trace(go.Scatter(x=t_epochs, y=t_winner_energy,
    name='Thrasher — winners (5, 21)',
    line=dict(color='rgba(80, 200, 120, 0.9)', width=2)))

fig.add_trace(go.Scatter(x=t_epochs, y=t_lost_energy,
    name='Thrasher — transient winner (2)',
    line=dict(color='rgba(255, 200, 50, 0.9)', width=1.5, dash='dot')))

fig.add_trace(go.Scatter(x=h_epochs, y=h_winner_energy,
    name='Healthy — winners (5, 15, 21)',
    line=dict(color='rgba(100, 160, 255, 0.7)', width=2, dash='dash')))

fig.add_vline(x=PLATEAU_EPOCH, line=dict(color='white', width=1.5, dash='dash'))

fig.update_layout(
    title='Competitor vs. Winner Energy Over Training',
    xaxis_title='Epoch',
    yaxis_title='Sum of QK^T fractions',
    height=420,
    template='plotly_dark',
    plot_bgcolor='#0a1010',
    paper_bgcolor='#060a0a',
    font=dict(family='IBM Plex Mono, monospace', size=11),
    legend=dict(x=1.0, y=0.99, xanchor='left'),
)
fig.show()

## 4. Attention Commitment → MLP Mass: The Lag

For freq 21 specifically: attention QK^T fraction vs. committed MLP neuron count.
The gap between these curves is the phenomenon the intervention is designed to shorten.

In [7]:
nd_epochs_t, nd_counts_t = load_neuron_counts(THRASHER_DIR)
nd_epochs_h, nd_counts_h = load_neuron_counts(HEALTHY_DIR)

TARGET_FREQ = 21   # freq label of interest
target_fi   = FREQ_LABELS.index(TARGET_FREQ)  # 0-based index in qk array

fig = sp.make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=['Thrasher (p=59, seed=485)', 'Healthy (p=59, seed=999)'],
    vertical_spacing=0.10,
    specs=[[{'secondary_y': True}], [{'secondary_y': True}]],
)

for row, (attn_epochs, qk_mean, nd_ep, nd_ct, label) in enumerate([
    (t_epochs, t_qk_mean, nd_epochs_t, nd_counts_t, 'Thrasher'),
    (h_epochs, h_qk_mean, nd_epochs_h, nd_counts_h, 'Healthy'),
], start=1):
    attn_series = qk_mean[:, target_fi]
    mlp_series  = nd_ct.get(TARGET_FREQ, [])

    fig.add_trace(go.Scatter(
        x=attn_epochs, y=attn_series,
        name=f'{label} — attn freq {TARGET_FREQ}',
        line=dict(color='rgba(255, 200, 50, 0.9)', width=2),
        legendgroup=f'{label}_attn',
    ), row=row, col=1, secondary_y=False)

    if mlp_series:
        fig.add_trace(go.Scatter(
            x=nd_ep, y=mlp_series,
            name=f'{label} — MLP neurons freq {TARGET_FREQ}',
            line=dict(color='rgba(80, 200, 120, 0.9)', width=2),
            legendgroup=f'{label}_mlp',
        ), row=row, col=1, secondary_y=True)

    fig.add_vline(x=PLATEAU_EPOCH, line=dict(color='white', width=1.5, dash='dash'),
                  row=row, col=1)  # type: ignore[arg-type]

fig.update_yaxes(title_text='QK^T fraction (attn)', secondary_y=False)
fig.update_yaxes(title_text='Committed neurons (MLP)', secondary_y=True)
fig.update_xaxes(title_text='Epoch', row=2, col=1)
fig.update_layout(
    title=f'Attention Commitment vs MLP Mass — freq {TARGET_FREQ}',
    height=700,
    template='plotly_dark',
    plot_bgcolor='#0a1010',
    paper_bgcolor='#060a0a',
    font=dict(family='IBM Plex Mono, monospace', size=11),
)
fig.show()

## 5. Intervention Config Sketch

Based on findings above, draft the gain vector for the first intervention experiment.

In [8]:
# Frequencies active at plateau onset, from dashboard data
plateau_profile = {
    2:  0.0318,   # transient winner
    4:  0.0357,   # competitor
    5:  0.0941,   # winner
    10: 0.0600,   # competitor
    12: 0.0773,   # competitor (strongest non-winner)
    17: 0.0130,   # competitor (weakest)
    21: 0.0364,   # winner (eventual dominant — but only ~5% in healthy model too at this epoch)
    22: 0.0878,   # competitor (strong)
}

# First pass: competitor dampening only.
# Hypothesis: clearing the noise floor is sufficient to let the model's own
# gradient signal resolve the competition. No boosting of winners — let the
# model decide what it wants to learn.

proposed_gain = {
    2:  1.0,   # leave — transient winner, observe naturally
    4:  0.3,   # dampen competitor
    5:  1.0,   # leave — strongest winner signal
    10: 0.3,   # dampen competitor
    12: 0.3,   # dampen competitor (strongest non-winner)
    17: 0.3,   # dampen competitor (weakest, but present)
    21: 1.0,   # leave — healthy model's freq 21 is similarly low at this epoch
    22: 0.3,   # dampen competitor (strong)
}

print('Proposed intervention config — first pass (competitor dampening only):')
print(f'  epoch_start:  {PLATEAU_EPOCH}')
print(f'  epoch_end:    {PLATEAU_EPOCH + 5000}  (adjust based on competition decay plot)')
print(f'  ramp_epochs:  200')
print(f'  target_heads: all')
print(f'  gain: {proposed_gain}')
print()
print('Rationale: dampening competitors (0.3x) without boosting winners.')
print('If this clears the competition window, that confirms failure mode #2.')
print('If competition persists, revisit with winner boosting in a follow-on run.')

Proposed intervention config — first pass (competitor dampening only):
  epoch_start:  1500
  epoch_end:    6500  (adjust based on competition decay plot)
  ramp_epochs:  200
  target_heads: all
  gain: {2: 1.0, 4: 0.3, 5: 1.0, 10: 0.3, 12: 0.3, 17: 0.3, 21: 1.0, 22: 0.3}

Rationale: dampening competitors (0.3x) without boosting winners.
If this clears the competition window, that confirms failure mode #2.
If competition persists, revisit with winner boosting in a follow-on run.


## 6. Run the Intervention

Three setup steps before training:
1. **Load W_E at plateau onset** — frequency directions are computed from the Thrasher's embedding weights at epoch 1500, capturing its own representational geometry at that moment
2. **Build frequency directions** — projects Fourier basis vectors through W_E into d_model space
3. **Construct hook** — wraps directions + gain config; the hook is stateless, called once per epoch

Training is a full fresh run (the intervention family retrains from scratch with the hook active during the window).

In [ ]:
from miscope import load_family
from miscope.families.implementations.frequency_gain_hook import (
    FrequencyGainHook,
    build_frequency_directions,
)

PRIME = 59
INTERVENTION_CONFIG = {
    "type": "frequency_gain",
    "target_heads": "all",
    "target_frequencies": [4, 10, 12, 17, 22],
    "gain": {4: 0.3, 10: 0.3, 12: 0.3, 17: 0.3, 22: 0.3},
    "epoch_start": 1500,
    "epoch_end": 6500,
    "ramp_epochs": 200,
}

# --- Step 1: load W_E from the Thrasher at plateau onset ---
baseline_family = load_family("modulo_addition_1layer")
thrasher = baseline_family.get_variant(prime=PRIME, seed=485, data_seed=598)
model_at_plateau = thrasher.load_model_at_checkpoint(PLATEAU_EPOCH)
W_E = model_at_plateau.embed.W_E.detach()
print(f"W_E shape: {W_E.shape}")  # expect (60, 128) — vocab=p+1, d_model=128

# --- Step 2: build frequency directions ---
ctx = thrasher.analysis_context()
fourier_basis = ctx["fourier_basis"]
D_sin, D_cos = build_frequency_directions(fourier_basis, W_E, prime=PRIME)
print(f"D_sin: {D_sin.shape}, D_cos: {D_cos.shape}")  # expect (29, 128) each

# --- Step 3: construct the hook ---
hook = FrequencyGainHook(INTERVENTION_CONFIG, D_sin, D_cos)

# --- Step 4: create the intervention variant ---
iv_family = load_family("modadd_intervention")
iv_variant = iv_family.create_intervention_variant(
    prime=PRIME, seed=485, data_seed=598,
    intervention_config=INTERVENTION_CONFIG,
)
print(f"Variant dir: {iv_variant.variant_dir}")

# --- Step 5: train ---
result = iv_variant.train(training_hook=hook)
print(f"\nTraining complete.")
print(f"  Final train loss: {result.train_losses[-1]:.6f}")
print(f"  Final test loss:  {result.test_losses[-1]:.6f}")
print(f"  Checkpoints saved: {len(result.checkpoint_epochs)}")